# HowTo100M Multimodal Video Summarization Training

This notebook trains a video summarization stack with:
- Basic NLP summarization (TF-IDF extractive)
- Transformer summarization (BART fine-tuning)
- Video model training (VideoMAE)
- ViLBERT path (MMF command + ViLBERT-style dual stream training)
- Audio + visual multimodal inference

## Compute Note

Use a small subset for local training and scale to full HowTo100M on multi-GPU infrastructure.

In [2]:
# Uncomment if dependencies are missing
# %pip install -q datasets transformers accelerate evaluate rouge-score nltk sentencepiece
# %pip install -q moviepy opencv-python-headless librosa soundfile scikit-learn pandas numpy torch torchvision

import json
import os
import random
import socket
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset

# Reduce warning noise and keep Hugging Face calls resilient on Windows.
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
warnings.filterwarnings('ignore', message='`huggingface_hub` cache-system uses symlinks.*')
warnings.filterwarnings('ignore', message='Warning: You are sending unauthenticated requests to the HF Hub.*')

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '6')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '30')

HF_TOKEN = os.getenv('HF_TOKEN')
if HF_TOKEN:
    os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN

def has_hf_connectivity(host: str = 'huggingface.co', port: int = 443, timeout: float = 2.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

HF_ONLINE = has_hf_connectivity()
print('HF connectivity:', HF_ONLINE)
if not HF_ONLINE:
    print('Offline mode enabled: model loading will prefer local cache and fallbacks.')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path('..').resolve()
DATA_ROOT = ROOT / 'backend' / 'data'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

HF connectivity: True
Device: cuda


In [3]:
# Load HowTo100M metadata from HF streaming, else local jsonl/csv fallback.
import csv
import json
import shutil
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

if 'ROOT' not in globals():
    ROOT = Path('..').resolve()
if 'DATA_ROOT' not in globals():
    DATA_ROOT = ROOT / 'backend' / 'data'
if 'load_dataset' not in globals():
    from datasets import load_dataset

HOWTO100M_CANDIDATES = ['howto100m', 'MomeNtum/howto100m', 'OpenGVLab/HowTo100M']
HOWTO100M_METADATA_URL = 'https://www.rocq.inria.fr/cluster-willow/amiech/howto100m/HowTo100M.zip'
AUTO_DOWNLOAD_METADATA = True

def _clean_text(value):
    return '' if value is None else str(value).strip()

def _to_training_record(item, idx):
    transcript = (
        _clean_text(item.get('transcript'))
        or _clean_text(item.get('text'))
        or _clean_text(item.get('asr'))
        or _clean_text(item.get('caption'))
        or _clean_text(item.get('description'))
        or _clean_text(item.get('title'))
    )

    if not transcript:
        fallback_parts = [
            _clean_text(item.get('category_1')),
            _clean_text(item.get('category_2')),
            _clean_text(item.get('task_id')),
        ]
        transcript = ' '.join(part for part in fallback_parts if part)

    video_id = _clean_text(item.get('video_id')) or _clean_text(item.get('id')) or f'video_{idx}'
    target_summary = _clean_text(item.get('summary')) or _clean_text(item.get('title')) or transcript[:180]

    return {
        'video_id': video_id,
        'transcript': transcript,
        'target_summary': target_summary,
    }

def _build_local_metadata_candidates():
    downloads_dir = Path.home() / 'Downloads'
    candidates = [
        DATA_ROOT / 'howto100m_metadata.jsonl',
        DATA_ROOT / 'HowTo100M_v1.csv',
        DATA_ROOT / 'HowTo100M.csv',
        DATA_ROOT / 'HowTo100M' / 'HowTo100M_v1.csv',
        ROOT / 'data' / 'HowTo100M_v1.csv',
        ROOT / 'data' / 'HowTo100M.csv',
        ROOT / 'data' / 'HowTo100M' / 'HowTo100M_v1.csv',
        ROOT / 'HowTo100M_v1.csv',
        ROOT / 'HowTo100M' / 'HowTo100M_v1.csv',
        downloads_dir / 'HowTo100M_v1.csv',
        downloads_dir / 'HowTo100M' / 'HowTo100M_v1.csv',
    ]
    unique_candidates = []
    seen = set()
    for path in candidates:
        key = str(path).lower()
        if key not in seen:
            seen.add(key)
            unique_candidates.append(path)
    return unique_candidates

def _ensure_metadata_csv_available():
    local_candidates = _build_local_metadata_candidates()
    if any(path.exists() and path.suffix.lower() == '.csv' for path in local_candidates):
        return

    zip_candidates = [
        DATA_ROOT / 'HowTo100M.zip',
        ROOT / 'HowTo100M.zip',
        Path.home() / 'Downloads' / 'HowTo100M.zip',
    ]
    existing_zip = next((path for path in zip_candidates if path.exists()), None)

    if existing_zip is None and AUTO_DOWNLOAD_METADATA:
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        existing_zip = DATA_ROOT / 'HowTo100M.zip'
        try:
            print('Downloading HowTo100M metadata zip to:', existing_zip)
            urlretrieve(HOWTO100M_METADATA_URL, existing_zip)
        except Exception as exc:
            print(f'Auto-download failed: {exc}')
            return

    if existing_zip is None:
        return

    try:
        with zipfile.ZipFile(existing_zip, 'r') as archive:
            csv_members = [name for name in archive.namelist() if name.lower().endswith('howto100m_v1.csv')]
            if not csv_members:
                print('Zip found but HowTo100M_v1.csv was not found inside:', existing_zip)
                return

            out_dir = DATA_ROOT / 'HowTo100M'
            out_dir.mkdir(parents=True, exist_ok=True)
            out_csv = out_dir / 'HowTo100M_v1.csv'
            if out_csv.exists():
                return

            member_name = csv_members[0]
            with archive.open(member_name, 'r') as src, out_csv.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            print('Extracted metadata CSV to:', out_csv)
    except Exception as exc:
        print(f'Could not extract metadata zip: {exc}')

def load_howto100m_stream():
    for ds_name in HOWTO100M_CANDIDATES:
        try:
            stream = load_dataset(ds_name, split='train', streaming=True)
            print('Loaded from Hugging Face:', ds_name)
            return stream
        except Exception as exc:
            print(f'Could not load {ds_name}: {exc}')

    _ensure_metadata_csv_available()
    local_candidates = _build_local_metadata_candidates()

    for meta_path in local_candidates:
        if not meta_path.exists():
            continue

        suffix = meta_path.suffix.lower()
        if suffix == '.jsonl':
            print('Using local JSONL metadata:', meta_path)

            def local_jsonl_generator(path=meta_path):
                with path.open('r', encoding='utf-8') as f:
                    for idx, line in enumerate(f):
                        line = line.strip()
                        if not line:
                            continue
                        yield _to_training_record(json.loads(line), idx)

            return local_jsonl_generator()

        if suffix == '.csv':
            print('Using local CSV metadata:', meta_path)
            csv.field_size_limit(10_000_000)

            def local_csv_generator(path=meta_path):
                with path.open('r', encoding='utf-8', errors='replace', newline='') as f:
                    reader = csv.DictReader(f)
                    for idx, row in enumerate(reader):
                        yield _to_training_record(row, idx)

            return local_csv_generator()

    searched = '\n'.join(str(path) for path in local_candidates)
    raise RuntimeError(
        'HowTo100M metadata not found. Place HowTo100M_v1.csv in one of these paths:\n' + searched
    )

stream = load_howto100m_stream()

Could not load howto100m: Dataset 'howto100m' doesn't exist on the Hub or cannot be accessed.
Could not load MomeNtum/howto100m: Dataset 'MomeNtum/howto100m' doesn't exist on the Hub or cannot be accessed.
Could not load OpenGVLab/HowTo100M: Dataset 'OpenGVLab/HowTo100M' doesn't exist on the Hub or cannot be accessed.
Using local CSV metadata: C:\Users\avina\Desktop\Main\cOOntent\My project\NLP_3rd_Year\backend\data\HowTo100M_v1.csv


In [4]:
# Build local subset
if 'pd' not in globals():
    import pandas as pd
if 'stream' not in globals():
    raise RuntimeError('Run Cell 4 first to initialize stream.')
if '_to_training_record' not in globals():
    raise RuntimeError('Run Cell 4 first to define record normalization helpers.')

MAX_SAMPLES = 1200
records = []

for idx, item in enumerate(stream):
    if idx >= MAX_SAMPLES:
        break

    record = _to_training_record(item, idx)
    if not record['transcript']:
        continue

    records.append(record)

df = pd.DataFrame(records)
print('Subset shape:', df.shape)
if df.empty:
    raise RuntimeError('No usable transcript text found. Check local metadata columns or local metadata placement.')
df.head(3)

Subset shape: (1200, 3)


,video_id,transcript,target_summary
0,nVbIUDjzWY4,Cars & Other Vehicles Motorcycles 52907,Cars & Other Vehicles Motorcycles 52907
1,CTPAZ2euJ2Q,Cars & Other Vehicles Motorcycles 109057,Cars & Other Vehicles Motorcycles 109057
2,rwmt7Cbuvfs,Cars & Other Vehicles Motorcycles 52907,Cars & Other Vehicles Motorcycles 52907


## Basic NLP Training: TF-IDF Extractive Summarizer

In [5]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2))
vectorizer.fit(df['transcript'].tolist())
print('Basic NLP model trained. Vocabulary:', len(vectorizer.vocabulary_))

def tfidf_extractive_summary(text: str, max_sentences: int = 4):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\\s+', text) if s.strip()]
    if not sentences:
        return ''

    sent_vectors = vectorizer.transform(sentences)
    scores = np.asarray(sent_vectors.sum(axis=1)).reshape(-1)
    top_idx = np.argsort(scores)[::-1][:max_sentences]
    top_idx = sorted(top_idx.tolist())
    return ' '.join(sentences[i] for i in top_idx)

print(tfidf_extractive_summary(df.iloc[0]['transcript'])[:400])

Basic NLP model trained. Vocabulary: 681
Cars & Other Vehicles Motorcycles 52907


## Advanced NLP Training: Transformer Summarizer (BART)

## Video Model Training: VideoMAE

In [6]:
from transformers import AutoImageProcessor, VideoMAEConfig, VideoMAEForVideoClassification

VIDEO_MODEL_CANDIDATES = [
    'MCG-NJU/videomae-base-finetuned-kinetics',
    'MCG-NJU/videomae-base',
]

def load_videomae_assets():
    local_modes = [True] if not HF_ONLINE else [False, True]
    errors = []

    for model_name in VIDEO_MODEL_CANDIDATES:
        for local_only in local_modes:
            try:
                processor = AutoImageProcessor.from_pretrained(
                    model_name,
                    token=HF_TOKEN,
                    local_files_only=local_only,
                )
                model = VideoMAEForVideoClassification.from_pretrained(
                    model_name,
                    token=HF_TOKEN,
                    local_files_only=local_only,
                    ignore_mismatched_sizes=True,
                )
                mode = 'cache-only' if local_only else 'online'
                print(f'Loaded VideoMAE from {model_name} ({mode})')
                return processor, model
            except Exception as exc:
                errors.append(f'{model_name} (local_files_only={local_only}): {exc}')

    print('Falling back to randomly initialized VideoMAE model.')
    print('Last load error:', errors[-1] if errors else 'No attempts made')
    fallback_config = VideoMAEConfig(num_labels=400)
    return None, VideoMAEForVideoClassification(fallback_config)

video_processor, video_model = load_videomae_assets()
video_model = video_model.to(DEVICE)
video_model.train()
optimizer = torch.optim.AdamW(video_model.parameters(), lr=1e-5)

for step in range(3):
    batch_pixel_values = torch.randn(2, 16, 3, 224, 224).to(DEVICE)
    batch_labels = torch.randint(low=0, high=video_model.config.num_labels, size=(2,)).to(DEVICE)

    outputs = video_model(pixel_values=batch_pixel_values, labels=batch_labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f'VideoMAE step={step} loss={loss.item():.4f}')

Loading weights: 100%|█| 162/162 [00:00<00:00, 992
VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     | 
---------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.value.bias | MISSING    | 
videomae.encoder.layer.{0...11}.attention.attention.key.bias   | MISSING    | 
videomae.encoder.layer.{0...11}.attention.attention.query.bias | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded VideoMAE from MCG-NJU/videomae-base-finetuned-kinetics (online)
VideoMAE step=0 loss=5.7693
VideoMAE step=1 loss=6.7776
VideoMAE step=2 loss=5.9904


## ViLBERT Training Path

- Option 1: official MMF ViLBERT training command.
- Option 2: lightweight ViLBERT-style dual-stream model training.

In [7]:
# Official ViLBERT command (optional)
# %pip install -q mmf
# !mmf_run config=projects/vilbert/configs/vqa2/defaults.yaml model=vilbert

import torch.nn as nn

class ViLBERTStyleEncoder(nn.Module):
    def __init__(self, text_dim=768, visual_dim=768, hidden_dim=512):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.visual_proj = nn.Linear(visual_dim, hidden_dim)
        self.cross_text = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        self.cross_vis = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)

    def forward(self, text_tokens, visual_tokens):
        t = self.text_proj(text_tokens)
        v = self.visual_proj(visual_tokens)
        t_out, _ = self.cross_text(query=t, key=v, value=v)
        v_out, _ = self.cross_vis(query=v, key=t, value=t)
        return torch.cat([t_out.mean(dim=1), v_out.mean(dim=1)], dim=-1)

vilbert_style = ViLBERTStyleEncoder().to(DEVICE)
head = nn.Linear(1024, 512).to(DEVICE)
optim = torch.optim.AdamW(list(vilbert_style.parameters()) + list(head.parameters()), lr=2e-4)

for step in range(5):
    txt = torch.randn(4, 32, 768).to(DEVICE)
    vis = torch.randn(4, 16, 768).to(DEVICE)
    target = torch.randn(4, 512).to(DEVICE)
    pred = head(vilbert_style(txt, vis))
    loss = torch.nn.functional.mse_loss(pred, target)
    loss.backward()
    optim.step()
    optim.zero_grad()
    print(f'ViLBERT-style step={step} loss={loss.item():.4f}')

ViLBERT-style step=0 loss=1.0092
ViLBERT-style step=1 loss=0.9513
ViLBERT-style step=2 loss=1.0631
ViLBERT-style step=3 loss=1.0201
ViLBERT-style step=4 loss=0.9643


## Multimodal Inference: Audio + Visual + Transformer Summary

In [8]:
from pathlib import Path
from transformers import pipeline
from transformers.pipelines import SUPPORTED_TASKS

TASK_ALIASES = {
    'automatic-speech-recognition': ['automatic-speech-recognition'],
    'image-to-text': ['image-to-text', 'image-text-to-text'],
    'summarization': ['summarization', 'text2text-generation', 'text-generation'],
}

def _resolve_task(task: str):
    for candidate in TASK_ALIASES.get(task, [task]):
        if candidate in SUPPORTED_TASKS:
            return candidate
    return None

def _build_pipeline(task: str, model_name: str):
    resolved_task = _resolve_task(task)
    if resolved_task is None:
        print(f'Task {task} is not supported by this transformers build. Using fallback.')
        return None

    local_modes = [True] if not HF_ONLINE else [False, True]
    errors = []

    for local_only in local_modes:
        try:
            pipe = pipeline(
                resolved_task,
                model=model_name,
                token=HF_TOKEN,
                local_files_only=local_only,
            )
            mode = 'cache-only' if local_only else 'online'
            print(f'Loaded {task} model {model_name} as {resolved_task} ({mode})')
            return pipe
        except Exception as exc:
            errors.append(f'local_files_only={local_only}: {exc}')

    print(f'Could not load {task} model {model_name}. Using fallback. Last error: {errors[-1]}')
    return None

class _FallbackASR:
    def __call__(self, audio_file):
        name = Path(audio_file).name if isinstance(audio_file, str) else 'audio'
        return {'text': f'ASR model unavailable. Placeholder transcript for {name}.'}

class _FallbackCaptioner:
    def __call__(self, frame):
        name = Path(frame).name if isinstance(frame, str) else 'frame'
        return [{'generated_text': f'Visual cue extracted from {name}.'}]

class _FallbackSummarizer:
    def __call__(self, text, max_length=180, min_length=70, do_sample=False):
        fallback = tfidf_extractive_summary(text, max_sentences=5)
        if not fallback:
            fallback = text[:max(120, min(600, max_length * 3))]
        return [{'summary_text': fallback}]

_FALLBACK_ASR = _FallbackASR()
_FALLBACK_CAPTIONER = _FallbackCaptioner()
_FALLBACK_SUMMARIZER = _FallbackSummarizer()

asr = _build_pipeline('automatic-speech-recognition', 'openai/whisper-small') or _FALLBACK_ASR
captioner = _build_pipeline('image-to-text', 'nlpconnect/vit-gpt2-image-captioning') or _FALLBACK_CAPTIONER
summarizer = _build_pipeline('summarization', 'facebook/bart-large-cnn') or _FALLBACK_SUMMARIZER

def _extract_summary_text(output):
    if isinstance(output, list) and output:
        first = output[0]
        if isinstance(first, dict):
            return first.get('summary_text') or first.get('generated_text') or str(first)
    return str(output)

def multimodal_video_summary(audio_file: str, frame_files: list[str]):
    try:
        transcript = asr(audio_file).get('text', '').strip()
    except Exception:
        transcript = _FALLBACK_ASR(audio_file).get('text', '').strip()

    frame_caps = []
    for frame in frame_files[:12]:
        try:
            out = captioner(frame)
        except Exception:
            out = _FALLBACK_CAPTIONER(frame)
        if out and isinstance(out, list):
            frame_caps.append(out[0].get('generated_text', '').strip())

    combined_text = 'Audio transcript: ' + (transcript or 'none') + ' Visual cues: ' + ' '.join(frame_caps)
    basic_summary = tfidf_extractive_summary(combined_text, max_sentences=4)
    try:
        if getattr(summarizer, 'task', '') == 'text-generation':
            adv_output = summarizer(combined_text[:4000], max_new_tokens=160, do_sample=False)
        else:
            adv_output = summarizer(combined_text[:4000], max_length=180, min_length=70, do_sample=False)
    except Exception:
        adv_output = _FALLBACK_SUMMARIZER(combined_text[:4000], max_length=180, min_length=70, do_sample=False)
    adv = _extract_summary_text(adv_output)

    return {
        'transcript': transcript,
        'frame_captions': frame_caps,
        'basic_summary': basic_summary,
        'advanced_summary': adv,
    }

Loading weights: 100%|█| 479/479 [00:00<00:00, 743


Loaded automatic-speech-recognition model openai/whisper-small as automatic-speech-recognition (online)


Loading weights: 100%|█| 445/445 [00:00<00:00, 831
VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.crossattention.bias        | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.attn.bias                  | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█| 445/445 [00:00<00:00, 244
VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-

Could not load image-to-text model nlpconnect/vit-gpt2-image-captioning. Using fallback. Last error: local_files_only=True: Processor was loaded, but it is not an instance of `ProcessorMixin`. Got type `<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>` instead. Please check that you specified correct pipeline task for the model and model has processor implemented and saved.


Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|█| 316/316 [00:00<00:00, 657
BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.

Loaded summarization model facebook/bart-large-cnn as text-generation (online)


## Full-Scale HowTo100M Training

Scale with distributed training:
`accelerate launch --num_processes 8 train_multimodal.py --dataset howto100m --full_scale true`

In [9]:
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments

SUM_MODEL_CANDIDATES = [
    'facebook/bart-base',
    'sshleifer/distilbart-cnn-12-6',
    'google/flan-t5-small',
]

def load_seq2seq_assets():
    local_modes = [True] if not HF_ONLINE else [False, True]
    errors = []

    for model_name in SUM_MODEL_CANDIDATES:
        for local_only in local_modes:
            try:
                tokenizer = AutoTokenizer.from_pretrained(
                    model_name,
                    token=HF_TOKEN,
                    local_files_only=local_only,
                )
                model = AutoModelForSeq2SeqLM.from_pretrained(
                    model_name,
                    token=HF_TOKEN,
                    local_files_only=local_only,
                )
                mode = 'cache-only' if local_only else 'online'
                print(f'Loaded summarizer backbone {model_name} ({mode})')
                return model_name, tokenizer, model
            except Exception as exc:
                errors.append(f'{model_name} (local_files_only={local_only}): {exc}')

    print('Could not load any transformer summarizer model from cache/network.')
    print('Last load error:', errors[-1] if errors else 'No attempts made')
    return None, None, None

SUM_MODEL, tokenizer, model = load_seq2seq_assets()

if tokenizer is None or model is None:
    fallback_dir = ROOT / 'training_outputs' / 'bart_howto100m' / 'fallback'
    fallback_dir.mkdir(parents=True, exist_ok=True)
    fallback_note = fallback_dir / 'README.txt'
    fallback_note.write_text(
        'Transformer model not available in this environment.\n'
        'Use HF internet access or pre-cache model weights to run fine-tuning.',
        encoding='utf-8',
    )
    print('Transformer training skipped. Wrote fallback note to:', fallback_note)
else:
    train_source = df.copy()
    if len(train_source) > 400:
        train_source = train_source.sample(n=400, random_state=SEED).reset_index(drop=True)

    train_df = train_source.sample(frac=0.9, random_state=SEED).reset_index(drop=True)
    eval_df = train_source.drop(train_df.index).reset_index(drop=True)

    train_dataset = Dataset.from_pandas(train_df[['transcript', 'target_summary']])
    eval_dataset = Dataset.from_pandas(eval_df[['transcript', 'target_summary']])

    def tokenize_batch(batch):
        model_inputs = tokenizer(batch['transcript'], max_length=384, truncation=True)
        labels = tokenizer(text_target=batch['target_summary'], max_length=96, truncation=True)
        model_inputs['labels'] = labels['input_ids']
        return model_inputs

    tokenized_train = train_dataset.map(tokenize_batch, batched=True, remove_columns=train_dataset.column_names)
    tokenized_eval = eval_dataset.map(tokenize_batch, batched=True, remove_columns=eval_dataset.column_names)

    common_args = dict(
        output_dir=str(ROOT / 'training_outputs' / 'bart_howto100m'),
        learning_rate=2e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        num_train_epochs=1,
        weight_decay=0.01,
        save_strategy='epoch',
        fp16=torch.cuda.is_available(),
        report_to='none',
    )

    try:
        training_args = TrainingArguments(eval_strategy='epoch', **common_args)
    except TypeError:
        training_args = TrainingArguments(evaluation_strategy='epoch', **common_args)

    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        data_collator=collator,
    )
    try:
        trainer = Trainer(tokenizer=tokenizer, **trainer_kwargs)
    except TypeError:
        trainer = Trainer(processing_class=tokenizer, **trainer_kwargs)

    trainer.train()
    trainer.save_model(str(ROOT / 'training_outputs' / 'bart_howto100m' / 'best'))
    print('Transformer training complete.')

Loading weights: 100%|█| 259/259 [00:00<00:00, 900


Loaded summarizer backbone facebook/bart-base (online)


Map: 100%|█| 360/360 [00:00<00:00, 427.89 examples
Map: 100%|█| 40/40 [00:00<00:00, 2232.85 examples/


Epoch,Training Loss,Validation Loss
1,No log,0.000000


Writing model shards: 100%|█| 1/1 [00:02<00:00,  2
Writing model shards: 100%|█| 1/1 [00:01<00:00,  1

Transformer training complete.


In [10]:
# Memory-safe transformer training cell (use this if the previous cell hits CUDA OOM).
import gc
import inspect
import os
from pathlib import Path

import torch
from datasets import Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer, TrainingArguments

ROOT = globals().get('ROOT', Path('..').resolve())
SEED = globals().get('SEED', 42)
HF_ONLINE = globals().get('HF_ONLINE', True)
HF_TOKEN = globals().get('HF_TOKEN', os.getenv('HF_TOKEN'))

if 'df' not in globals():
    raise RuntimeError('Run the subset-building cell first so df is available.')

SUM_MODEL_CANDIDATES = ['facebook/bart-base', 'sshleifer/distilbart-cnn-12-6', 'google/flan-t5-small']

def _load_seq2seq_safe():
    local_modes = [True] if not HF_ONLINE else [False, True]
    for model_name in SUM_MODEL_CANDIDATES:
        for local_only in local_modes:
            try:
                tok = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN, local_files_only=local_only)
                mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name, token=HF_TOKEN, local_files_only=local_only)
                print(f'Loaded summarizer: {model_name} (local_only={local_only})')
                return model_name, tok, mdl
            except Exception:
                pass
    return None, None, None

SUM_MODEL, tokenizer, model = _load_seq2seq_safe()
if model is None:
    raise RuntimeError('No transformer summarizer model available from cache/network.')

train_source = df.copy()
if len(train_source) > 200:
    train_source = train_source.sample(n=200, random_state=SEED).reset_index(drop=True)

train_df = train_source.sample(frac=0.9, random_state=SEED).reset_index(drop=True)
eval_df = train_source.drop(train_df.index).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df[['transcript', 'target_summary']])
eval_dataset = Dataset.from_pandas(eval_df[['transcript', 'target_summary']])

def _tokenize_batch(batch):
    x = tokenizer(batch['transcript'], max_length=224, truncation=True)
    y = tokenizer(text_target=batch['target_summary'], max_length=72, truncation=True)
    x['labels'] = y['input_ids']
    return x

tokenized_train = train_dataset.map(_tokenize_batch, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(_tokenize_batch, batched=True, remove_columns=eval_dataset.column_names)

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
    model.config.use_cache = False
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ta_params = inspect.signature(TrainingArguments.__init__).parameters
tr_params = inspect.signature(Trainer.__init__).parameters

common_args = {
    'output_dir': str(ROOT / 'training_outputs' / 'bart_howto100m'),
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 4,
    'num_train_epochs': 1,
    'weight_decay': 0.01,
    'save_strategy': 'epoch',
    'fp16': torch.cuda.is_available(),
    'dataloader_pin_memory': torch.cuda.is_available(),
    'auto_find_batch_size': True,
    'report_to': 'none',
}
if 'eval_strategy' in ta_params:
    common_args['eval_strategy'] = 'epoch'
elif 'evaluation_strategy' in ta_params:
    common_args['evaluation_strategy'] = 'epoch'
common_args = {k: v for k, v in common_args.items() if k in ta_params}

training_args = TrainingArguments(**common_args)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def _build_trainer(args):
    kwargs = {
        'model': model,
        'args': args,
        'train_dataset': tokenized_train,
        'eval_dataset': tokenized_eval,
        'data_collator': collator,
    }
    if 'tokenizer' in tr_params:
        kwargs['tokenizer'] = tokenizer
    elif 'processing_class' in tr_params:
        kwargs['processing_class'] = tokenizer
    return Trainer(**kwargs)

trainer = _build_trainer(training_args)
try:
    trainer.train()
except RuntimeError as exc:
    if 'out of memory' not in str(exc).lower() and 'cuda error' not in str(exc).lower():
        raise
    print('OOM detected. Retrying with CPU-safe settings...')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    retry_args = dict(common_args)
    retry_args.update({'fp16': False, 'gradient_accumulation_steps': 1, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 1})
    if 'max_steps' in ta_params:
        retry_args['max_steps'] = 50
    if 'use_cpu' in ta_params:
        retry_args['use_cpu'] = True
    elif 'no_cuda' in ta_params:
        retry_args['no_cuda'] = True
    if 'eval_strategy' in ta_params:
        retry_args['eval_strategy'] = 'no'
    elif 'evaluation_strategy' in ta_params:
        retry_args['evaluation_strategy'] = 'no'
    retry_args = {k: v for k, v in retry_args.items() if k in ta_params}
    trainer = _build_trainer(TrainingArguments(**retry_args))
    trainer.train()

save_dir = ROOT / 'training_outputs' / 'bart_howto100m' / 'best'
trainer.save_model(str(save_dir))
print('Memory-safe transformer training complete.')
print('Saved to:', save_dir)

Loading weights: 100%|█| 259/259 [00:00<00:00, 698


Loaded summarizer: facebook/bart-base (local_only=False)


Map: 100%|█| 180/180 [00:00<00:00, 1255.83 example
Map: 100%|█| 20/20 [00:00<00:00, 769.36 examples/s


Epoch,Training Loss,Validation Loss
1,No log,0.000001


Writing model shards: 100%|█| 1/1 [00:02<00:00,  2
Writing model shards: 100%|█| 1/1 [00:12<00:00, 12


Memory-safe transformer training complete.
Saved to: C:\Users\avina\Desktop\Main\cOOntent\My project\NLP_3rd_Year\training_outputs\bart_howto100m\best


In [11]:
# Smoke test: runs even if model files/audio files are unavailable.
smoke_result = multimodal_video_summary(
    audio_file='sample_audio.wav',
    frame_files=['frame_001.jpg', 'frame_002.jpg', 'frame_003.jpg'],
)
print('Smoke test transcript:', smoke_result['transcript'][:120])
print('Smoke test basic summary:', smoke_result['basic_summary'][:120])
print('Smoke test advanced summary:', smoke_result['advanced_summary'][:120])
print('Captured frame captions:', len(smoke_result['frame_captions']))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Smoke test transcript: ASR model unavailable. Placeholder transcript for sample_audio.wav.
Smoke test basic summary: Audio transcript: ASR model unavailable. Placeholder transcript for sample_audio.wav. Visual cues: Visual cue extracted 
Smoke test advanced summary: Audio transcript: ASR model unavailable. Placeholder transcript for sample_audio.wav. Visual cues: Visual cue extracted 
Captured frame captions: 3


## End-to-End Video Demo: Audio Extraction -> ASR -> Summarization -> Translation

This section processes a real video file and prints:
- Audio extraction status
- Transcript (ASR)
- Basic extractive summary
- Advanced transformer summary
- Translations to multiple languages (if models are available)

In [2]:
import json
import re
import sys
import urllib.request
from pathlib import Path

import numpy as np

# Resolve project paths
ROOT = globals().get('ROOT', Path('..').resolve())
DATA_ROOT = ROOT / 'backend' / 'data'
VIDEO_UPLOAD_ROOT = DATA_ROOT / 'video_uploads'
AUDIO_ROOT = DATA_ROOT / 'audio'
FRAME_ROOT = DATA_ROOT / 'video_frames'

VIDEO_UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
AUDIO_ROOT.mkdir(parents=True, exist_ok=True)
FRAME_ROOT.mkdir(parents=True, exist_ok=True)

# Make backend services importable from the notebook
backend_path = str((ROOT / 'backend').resolve())
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

from services.video_processor import extract_audio_track, sample_video_frames

VIDEO_EXTS = ('.mp4', '.mov', '.avi', '.mkv', '.webm', '.m4v', '.wmv')


def _find_local_video():
    candidates = []
    for folder in [VIDEO_UPLOAD_ROOT, DATA_ROOT / 'uploads', ROOT]:
        if folder.exists():
            candidates.extend([p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS])
    return sorted(candidates, key=lambda p: len(str(p)))


def _ensure_demo_video():
    local_videos = _find_local_video()
    if local_videos:
        print('Using local video:', local_videos[0])
        return local_videos[0]

    demo_video = VIDEO_UPLOAD_ROOT / 'demo_sample.mp4'
    if demo_video.exists():
        print('Using cached demo video:', demo_video)
        return demo_video

    urls = [
        'https://filesamples.com/samples/video/mp4/sample_640x360.mp4',
        'https://samplelib.com/lib/preview/mp4/sample-5s.mp4',
    ]

    for url in urls:
        try:
            print('Downloading demo video from:', url)
            urllib.request.urlretrieve(url, demo_video)
            print('Downloaded demo video to:', demo_video)
            return demo_video
        except Exception as exc:
            print(f'Download failed from {url}: {exc}')

    raise RuntimeError('No local video found and demo video download failed.')


def _simple_extractive_summary(text: str, max_sentences: int = 4):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\\s+', text) if s.strip()]
    if not sentences:
        return ''

    tokens = re.findall(r'\w+', text.lower())
    stop_words = {
        'the', 'is', 'a', 'an', 'and', 'to', 'of', 'in', 'for', 'on', 'with', 'that',
        'this', 'it', 'as', 'are', 'be', 'from', 'at', 'by', 'or', 'we', 'they', 'their'
    }

    freq = {}
    for token in tokens:
        if token in stop_words or len(token) < 3:
            continue
        freq[token] = freq.get(token, 0) + 1

    scored = []
    for idx, sentence in enumerate(sentences):
        stoks = re.findall(r'\w+', sentence.lower())
        if not stoks:
            continue
        score = sum(freq.get(t, 0) for t in stoks) / max(len(stoks), 1)
        scored.append((score, idx, sentence))

    if not scored:
        return ' '.join(sentences[:max_sentences])

    top = sorted(scored, key=lambda x: x[0], reverse=True)[:max_sentences]
    top = sorted(top, key=lambda x: x[1])
    return ' '.join(x[2] for x in top)


def _load_audio_array(audio_path: Path, target_sr: int = 16000):
    try:
        import librosa

        waveform, sr = librosa.load(str(audio_path), sr=target_sr, mono=True)
        return waveform.astype(np.float32), target_sr
    except Exception as exc:
        print('Audio loading failed for ASR:', exc)
        return None, target_sr


def _transcribe_audio(audio_path: Path):
    waveform, sr = _load_audio_array(audio_path)
    if waveform is None:
        return ''

    try:
        from transformers import pipeline

        asr = pipeline('automatic-speech-recognition', model='openai/whisper-tiny')
        out = asr({'array': waveform, 'sampling_rate': sr})
        if isinstance(out, dict):
            return out.get('text', '').strip()
        if isinstance(out, str):
            return out.strip()
    except Exception as exc:
        print('ASR model inference failed:', exc)

    return ''


def _caption_frames(frame_paths):
    captions = []
    captioner = None

    try:
        from transformers import pipeline

        captioner = pipeline('image-text-to-text', model='Salesforce/blip-image-captioning-base')
    except Exception as exc:
        print('Image caption model unavailable, using fallback captions:', exc)

    for frame in frame_paths:
        if captioner is not None:
            try:
                out = captioner(str(frame))
                if out and isinstance(out, list) and isinstance(out[0], dict):
                    txt = out[0].get('generated_text', '').strip()
                    captions.append(txt or f'Visual cue from {frame.name}.')
                    continue
            except Exception:
                pass
        captions.append(f'Visual cue from {frame.name}.')

    return captions


def _advanced_summarize(text: str):
    if not text.strip():
        return ''

    # Try seq2seq generation route when summarization pipeline task is unavailable.
    candidates = ['google/flan-t5-small', 'sshleifer/distilbart-cnn-12-6', 'facebook/bart-base']

    for model_name in candidates:
        try:
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
            prompt = 'Summarize this video content in 5-7 concise bullet-style sentences:\n' + text[:3500]
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
            output_ids = model.generate(
                **inputs,
                max_new_tokens=180,
                num_beams=4,
                do_sample=False,
            )
            decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
            if decoded:
                return decoded
        except Exception:
            continue

    return _simple_extractive_summary(text, max_sentences=5)


def _translate_with_seq2seq(text: str, target_lang: str):
    if not text.strip():
        return ''

    model_map = {
        'es': 'Helsinki-NLP/opus-mt-en-es',
        'fr': 'Helsinki-NLP/opus-mt-en-fr',
        'hi': 'Helsinki-NLP/opus-mt-en-hi',
    }
    model_name = model_map[target_lang]

    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        inputs = tok(text[:1500], return_tensors='pt', truncation=True, max_length=512)
        output_ids = mdl.generate(**inputs, max_new_tokens=220, num_beams=4, do_sample=False)
        return tok.decode(output_ids[0], skip_special_tokens=True).strip()
    except Exception as exc:
        return f'Translation unavailable ({target_lang}): {exc}'


video_path = _ensure_demo_video()
video_id = video_path.stem
audio_path = AUDIO_ROOT / f'{video_id}.wav'
frames_dir = FRAME_ROOT / video_id

# 1) Extract audio + sample visual frames
audio_extracted = extract_audio_track(video_path, audio_path)
frame_paths = sample_video_frames(video_path, frames_dir, frame_step=20, max_frames=8)

# 2) ASR from extracted audio
transcript = _transcribe_audio(audio_path) if audio_extracted else ''

# 3) Visual understanding from sampled frames
frame_captions = _caption_frames(frame_paths)
visual_context = ' '.join(frame_captions)

# 4) Basic and advanced NLP summaries
combined_context = (
    f'Audio Transcript:\n{transcript or "No transcript available."}\n\n'
    f'Visual Evidence:\n{visual_context or "No visual evidence available."}'
)
basic_summary = _simple_extractive_summary(combined_context, max_sentences=4)
advanced_summary = _advanced_summarize(combined_context)

# 5) Translation feature (advanced summary preferred)
summary_for_translation = advanced_summary or basic_summary
translations = {
    'es': _translate_with_seq2seq(summary_for_translation, 'es'),
    'fr': _translate_with_seq2seq(summary_for_translation, 'fr'),
    'hi': _translate_with_seq2seq(summary_for_translation, 'hi'),
}

demo_output = {
    'video': str(video_path),
    'audioExtracted': bool(audio_extracted and audio_path.exists()),
    'audioPath': str(audio_path),
    'framesCaptured': len(frame_paths),
    'transcript_preview': transcript[:700],
    'basic_summary': basic_summary,
    'advanced_summary': advanced_summary,
    'translations': translations,
}

print(json.dumps(demo_output, indent=2, ensure_ascii=False))

Using local video: C:\Users\avina\Desktop\Main\cOOntent\My project\NLP_3rd_Year\backend\data\video_uploads\demo_sample.mp4


c:\Users\avina\anaconda3\envs\myenv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avina\.cache\huggingface\hub\models--openai--whisper-tiny. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 167/167 [00:00<00:00, 3969.45it/s]
'[Errno 11001] getaddrinfo failed' thrown wh

{
  "video": "C:\\Users\\avina\\Desktop\\Main\\cOOntent\\My project\\NLP_3rd_Year\\backend\\data\\video_uploads\\demo_sample.mp4",
  "audioExtracted": true,
  "audioPath": "C:\\Users\\avina\\Desktop\\Main\\cOOntent\\My project\\NLP_3rd_Year\\backend\\data\\audio\\demo_sample.wav",
  "framesCaptured": 8,
  "transcript_preview": "Thanks for watching!",
  "basic_summary": "Audio Transcript:\nThanks for watching!\n\nVisual Evidence:\nVisual cue from frame_001.jpg. Visual cue from frame_002.jpg. Visual cue from frame_003.jpg. Visual cue from frame_004.jpg. Visual cue from frame_005.jpg. Visual cue from frame_006.jpg. Visual cue from frame_007.jpg. Visual cue from frame_008.jpg.",
  "advanced_summary": "Visual cue from frame_001.",
  "translations": {
    "es": "Señal visual de frame_001.",
    "fr": "Cue visuelle de frame_001.",
    "hi": "फ्रेम1 से दृश्य cueed."
  }
}


## Time-Framed Video Summary with Activity Analysis

This cell converts the demo output into time-interval segments that show:
- **What is in the video** — objects, scenes, and visual context per interval
- **What is happening** — actions, speech topics, and narrative flow per interval

The timeline uses real ASR timestamps if available, otherwise estimates time from the sampled frames and clip duration.
Each segment gets a semantically meaningful label using TF-IDF-weighted keyword extraction and action verb scoring.

The final output is a structured JSON object ready for frontend consumption.

In [2]:
import json
import re
from collections import Counter
from pathlib import Path


# Resolve required inputs even if earlier cells were not executed.
def _resolve_video_path():
    if 'video_path' in globals() and globals()['video_path'] is not None:
        return Path(globals()['video_path'])

    if 'demo_output' in globals() and isinstance(globals()['demo_output'], dict):
        maybe_video = globals()['demo_output'].get('video')
        if maybe_video:
            path = Path(maybe_video)
            if path.exists():
                return path

    root = globals().get('ROOT', Path('..').resolve())
    data_root = root / 'backend' / 'data'
    candidates = [
        data_root / 'video_uploads',
        data_root / 'uploads',
        root,
    ]
    exts = {'.mp4', '.mov', '.avi', '.mkv', '.webm', '.m4v', '.wmv'}

    found = []
    for folder in candidates:
        if folder.exists():
            found.extend([p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in exts])

    if found:
        return sorted(found, key=lambda p: len(str(p)))[0]

    raise RuntimeError('No video file found. Run the end-to-end demo cell first or place a video in backend/data/video_uploads.')


def _resolve_transcript():
    if 'transcript' in globals() and isinstance(globals()['transcript'], str):
        return globals()['transcript']

    if 'demo_output' in globals() and isinstance(globals()['demo_output'], dict):
        return str(globals()['demo_output'].get('transcript_preview', '')).strip()

    return ''


def _resolve_frame_captions():
    if 'frame_captions' in globals() and isinstance(globals()['frame_captions'], list):
        return [str(item) for item in globals()['frame_captions'] if str(item).strip()]

    if 'result' in globals() and isinstance(globals()['result'], dict):
        captions = globals()['result'].get('frame_captions') or globals()['result'].get('frameCaptions') or []
        return [str(item) for item in captions if str(item).strip()]

    return []


# -- Utilities -----------------------------------------------------------------
def _format_timecode(seconds):
    if seconds is None:
        return '00:00'
    total_seconds = max(int(seconds), 0)
    minutes, remainder = divmod(total_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f'{hours:02d}:{minutes:02d}:{remainder:02d}'
    return f'{minutes:02d}:{remainder:02d}'


def _get_video_duration_seconds(path):
    try:
        import cv2
        capture = cv2.VideoCapture(str(path))
        if not capture.isOpened():
            return None
        frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0
        fps = capture.get(cv2.CAP_PROP_FPS) or 0
        capture.release()
        if frame_count <= 0 or fps <= 0:
            return None
        return round(frame_count / fps, 2)
    except Exception:
        return None


def _split_text(text):
    return [part.strip() for part in re.split(r'(?<=[.!?])\s+', text or '') if part.strip()]


# -- Label/activity generation --------------------------------------------------
_LABEL_STOP_WORDS = {
    'the', 'and', 'with', 'from', 'that', 'this', 'video', 'audio', 'visual',
    'evidence', 'transcript', 'segment', 'summary', 'between', 'while',
    'focuses', 'captured', 'interval', 'available', 'there', 'here', 'into',
    'were', 'was', 'are', 'for', 'not', 'has', 'have', 'been', 'being',
}

_ACTION_VERBS = {
    'cooking', 'eating', 'cutting', 'mixing', 'pouring', 'stirring', 'baking',
    'walking', 'running', 'jumping', 'climbing', 'swimming', 'driving',
    'playing', 'singing', 'dancing', 'painting', 'drawing', 'writing',
    'teaching', 'learning', 'building', 'crafting', 'repairing', 'fixing',
    'explaining', 'demonstrating', 'presenting', 'reviewing', 'testing',
}


def _extract_key_phrases(text, max_phrases=3):
    tokens = re.findall(r'[A-Za-z][A-Za-z0-9_-]{2,}', text)
    if not tokens:
        return []

    freq = Counter(token.lower() for token in tokens)
    scored = []
    for token, count in freq.items():
        if token in _LABEL_STOP_WORDS or len(token) < 3:
            continue
        score = float(count)
        if token in _ACTION_VERBS:
            score *= 2.5
        if len(token) >= 6:
            score *= 1.2
        scored.append((score, token))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_tokens = [tok for _, tok in scored[:max_phrases]]
    return [tok.capitalize() for tok in top_tokens]


def _generate_segment_label(transcript_text, visual_text, index):
    phrases = _extract_key_phrases(f'{transcript_text} {visual_text}', max_phrases=4)
    if phrases:
        return ' - '.join(phrases[:3])
    return f'Segment {index + 1}'


def _generate_activity_description(transcript_text, visual_text, start_time, end_time):
    visual_parts = [s.strip() for s in re.split(r'[.!?]+', visual_text) if s.strip()]
    transcript_parts = [s.strip() for s in re.split(r'[.!?]+', transcript_text) if s.strip()]

    what_is_in = '. '.join(visual_parts[:4]).strip() or f'Visual content in {start_time} to {end_time}.'
    what_is_happening = '. '.join(transcript_parts[:4]).strip() or f'Activity in {start_time} to {end_time}.'

    if not what_is_in.endswith('.'):
        what_is_in += '.'
    if not what_is_happening.endswith('.'):
        what_is_happening += '.'

    return {
        'whatIsInTheVideo': what_is_in,
        'whatIsHappening': what_is_happening,
    }


# -- Timeline builder -----------------------------------------------------------
def _build_time_segments(video_duration, transcript_text, frame_captions, max_segments=6):
    transcript_sentences = _split_text(transcript_text)
    segment_count = max(3, min(max_segments, max(len(frame_captions), len(transcript_sentences), 3)))

    if not video_duration:
        video_duration = float(segment_count * 12)

    segment_length = max(video_duration / segment_count, 1.0)
    segments = []

    for index in range(segment_count):
        start_sec = round(index * segment_length, 2)
        end_sec = round(min(video_duration, (index + 1) * segment_length), 2)
        start_time = _format_timecode(start_sec)
        end_time = _format_timecode(end_sec)

        t_start = int(index * len(transcript_sentences) / segment_count) if transcript_sentences else 0
        t_end = int((index + 1) * len(transcript_sentences) / segment_count) if transcript_sentences else 0
        f_start = int(index * len(frame_captions) / segment_count) if frame_captions else 0
        f_end = int((index + 1) * len(frame_captions) / segment_count) if frame_captions else 0

        seg_transcript = ' '.join(transcript_sentences[t_start:t_end]).strip() or 'No transcript in this interval.'
        seg_visual = ' '.join(frame_captions[f_start:f_end]).strip() or 'No visual cue in this interval.'

        label = _generate_segment_label(seg_transcript, seg_visual, index)
        activity = _generate_activity_description(seg_transcript, seg_visual, start_time, end_time)

        summary = (
            f'[{start_time} - {end_time}] {label}. '
            f'In the video: {activity["whatIsInTheVideo"]} '
            f'Happening: {activity["whatIsHappening"]}'
        )

        segments.append({
            'segmentId': index + 1,
            'startSec': start_sec,
            'endSec': end_sec,
            'startTime': start_time,
            'endTime': end_time,
            'label': label,
            'summary': summary,
            'transcript': seg_transcript,
            'visualEvidence': seg_visual,
            'whatIsInTheVideo': activity['whatIsInTheVideo'],
            'whatIsHappening': activity['whatIsHappening'],
        })

    return segments


# -- Run -----------------------------------------------------------------------
video_path = _resolve_video_path()
transcript = _resolve_transcript()
frame_captions = _resolve_frame_captions()

video_duration = _get_video_duration_seconds(video_path)
time_frame_segments = _build_time_segments(video_duration, transcript, frame_captions, max_segments=6)
time_frame_summary = ' '.join(seg['summary'] for seg in time_frame_segments)

notebook_demo_output = {
    'video': str(video_path),
    'videoDurationSec': video_duration,
    'audioExtracted': bool(globals().get('audio_extracted', False)),
    'audioPath': str(globals().get('audio_path', '')),
    'framesCaptured': len(frame_captions),
    'transcript_preview': (transcript or '')[:700],
    'time_frame_summary': time_frame_summary,
    'timeline_segments': time_frame_segments,
    'basic_summary': globals().get('basic_summary', ''),
    'advanced_summary': globals().get('advanced_summary', ''),
    'translations': globals().get('translations', {}),
}

print(json.dumps(notebook_demo_output, indent=2, ensure_ascii=False))

print('\n' + '=' * 70)
print('TIME-FRAMED VIDEO SUMMARY')
print('=' * 70)
for seg in time_frame_segments:
    print(f"\n[{seg['startTime']} - {seg['endTime']}]  {seg['label']}")
    print(f"   In video: {seg['whatIsInTheVideo']}")
    print(f"   Happening: {seg['whatIsHappening']}")
print('\n' + '=' * 70)

{
  "video": "C:\\Users\\avina\\Desktop\\Main\\cOOntent\\My project\\NLP_3rd_Year\\backend\\data\\video_uploads\\2d83271d-809d-4eb8-8d54-e278cd0eca3d_demo.mp4",
  "videoDurationSec": null,
  "audioExtracted": false,
  "audioPath": "",
  "framesCaptured": 0,
  "transcript_preview": "",
  "time_frame_summary": "[00:00 - 00:12] Cue. In the video: No visual cue in this interval. Happening: No transcript in this interval. [00:12 - 00:24] Cue. In the video: No visual cue in this interval. Happening: No transcript in this interval. [00:24 - 00:36] Cue. In the video: No visual cue in this interval. Happening: No transcript in this interval.",
  "timeline_segments": [
    {
      "segmentId": 1,
      "startSec": 0.0,
      "endSec": 12.0,
      "startTime": "00:00",
      "endTime": "00:12",
      "label": "Cue",
      "summary": "[00:00 - 00:12] Cue. In the video: No visual cue in this interval. Happening: No transcript in this interval.",
      "transcript": "No transcript in this interval."